# Random rollout on CartPole

This notebook shows the smallest end-to-end loop: build an environment from `EnvConfig`, sample random actions, and watch the rollout as an embedded MP4.

## What mouse-gym does

Most Gymnasium environments are **episodic**: an agent acts until termination or truncation, then the caller must call `reset()` before the next episode can begin. That interface works well when each episode is an independent sample. It becomes awkward when the experiment is about behavior *across* multiple episodes — where what an agent learns or observes in one episode should influence the next.

**mouse-gym** converts episodic Gymnasium environments into **continuing environments**. You call `step()` in a loop, forever. When an episode ends, mouse-gym resets the underlying environment internally and emits the first observation of the next episode on the very next step — no public `reset()` call required, no special handling needed at the episode boundary.

Three design decisions make this more than a thin wrapper:

- **Reset frames are first-class outputs.** The reset observation appears in the same output record shape as every other step, with `time=0`, `done=0`, and the configured `reset_reward`. Training code sees a uniform stream.
- **Episode structure stays visible.** The `done` field uses integer codes so transitions, truncations, and reset frames are all distinguishable: `0` = running, `1` = terminated (Gymnasium `terminated=True`), `2` = truncated (Gymnasium `truncated=True`). Reset frames always emit `done=0`.

## Step outputs

Each `step()` returns one output `dict`:

| Field | Type | Notes |
|-------|------|-------|
| `observation` | tensor | Native shape from the env (e.g. `(4,)` for CartPole). |
| `reward` | float32 tensor | Raw env reward. Uses `reset_reward` (default `0`) on reset frames. |
| `done` | int64 tensor | `0` running · `1` ep. terminated · `2` ep. truncated · `3` task terminated · `4` task truncated. |
| `time` | int64 tensor | Step index within the current episode (0-based; `0` on reset frames). |
| `episode_index` | int | Episode counter. |
| `task_index` | int | Task counter. |

## Introspecting contracts: `input_spec` and `output_spec`

`env.input_spec` and `env.output_spec` describe the stable construction-time contract. The Gymnasium `info` dict is forwarded verbatim on each step output under `info`.

```python
ispec = env.input_spec
ospec = env.output_spec

ispec.action.dtype   # torch.int64 (discrete) or torch.float32 (continuous)
ispec.action.shape   # () for scalar; (n,) for multi-dimensional

ospec.observation.dtype   # torch.float32 for CartPole
ospec.observation.shape   # (4,) for CartPole
ospec.reward.dtype        # torch.float32
```

Use these to allocate replay buffers, build network heads, or validate shapes before running any rollout.

## Imports

In [1]:
import os

import matplotlib.animation as animation
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# Some Gymnasium envs render via pygame; run headless when no display is available.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

from mouse_gym import EnvConfig, make_env, make_group_env

## Configure the environment

`EnvConfig` is the configuration object passed to `make_env`. One config creates one `SingleEnv`; pass a list to `make_group_env` for multiple instances. Required fields:

| Field | Purpose |
|-------|---------|
| `id` | Gymnasium env id |
| `reset_seed` | Seed for mouse-gym's internal Gymnasium reset stream |
| `episodes_per_task` | Number of episodes before a task boundary (`done=3/4`) fires |

Everything else is optional. Notable optional fields: `name` overrides `env.name`; `kwargs` passes arguments to `gym.make`; `env_fn` replaces id-based construction with a factory (see notebook 02).

`make_env` returns a `SingleEnv`. `env.name` is the env instance name — e.g. `"CartPole-v1"` here when `name` is not set.

In [2]:
cfg = EnvConfig(
    id="CartPole-v1",
    reset_seed=0,
    episodes_per_task=100,
    kwargs={"render_mode": "rgb_array"},
)
env = make_env(cfg)

## Rollout

Run CartPole for 200 random-action steps. There is no public `reset()` call — `step()` handles episode boundaries internally. Frames are captured each step and displayed as an embedded MP4 at the end.

Each iteration prints the action dict (`inp`) and the full output dict (`output`) so you can see the `done`, `time`, `reward`, and `episode_index` values live. Watch `time` reset to `0` and `done` go non-zero whenever CartPole terminates.

In [3]:
frames = []
for _ in range(200):
    inp = env.sample_random_input()
    output = env.step(inp)
    frames.append(env.render()[0])
    print(inp)
    print(output)


def display_env_replay(name, frames, *, fps=20):
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.axis("off")
    ax.set_title(name, fontsize=10)
    img = ax.imshow(frames[0])

    def update(t):
        img.set_data(frames[t])
        return (img,)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / fps,
        blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_html5_video()))


display_env_replay(env.name, frames)

/home/user/Repos/mouse-gym/.venv/lib/python3.12/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


{'action': tensor(0)}
{'time': tensor(0), 'reward': tensor(0.), 'done': tensor(0), 'episode_index': 0, 'task_index': 0, 'observation': tensor([-0.0147,  0.0180,  0.0376, -0.0261])}
{'action': tensor(0)}
{'time': tensor(1), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 0, 'task_index': 0, 'observation': tensor([-0.0143, -0.1776,  0.0370,  0.2781])}
{'action': tensor(0)}
{'time': tensor(2), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 0, 'task_index': 0, 'observation': tensor([-0.0179, -0.3733,  0.0426,  0.5823])}
{'action': tensor(0)}
{'time': tensor(3), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 0, 'task_index': 0, 'observation': tensor([-0.0253, -0.5690,  0.0543,  0.8881])}
{'action': tensor(0)}
{'time': tensor(4), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 0, 'task_index': 0, 'observation': tensor([-0.0367, -0.7648,  0.0720,  1.1973])}
{'action': tensor(1)}
{'time': tensor(5), 'reward': tensor(1.), 'done': tensor(0), 'episode_ind

{'action': tensor(1)}
{'time': tensor(16), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 3, 'task_index': 0, 'observation': tensor([ 0.1216,  0.3949, -0.1571, -0.9197])}
{'action': tensor(0)}
{'time': tensor(17), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 3, 'task_index': 0, 'observation': tensor([ 0.1295,  0.2022, -0.1754, -0.6802])}
{'action': tensor(0)}
{'time': tensor(18), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 3, 'task_index': 0, 'observation': tensor([ 0.1335,  0.0099, -0.1891, -0.4475])}
{'action': tensor(0)}
{'time': tensor(19), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 3, 'task_index': 0, 'observation': tensor([ 0.1337, -0.1821, -0.1980, -0.2198])}
{'action': tensor(1)}
{'time': tensor(20), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 3, 'task_index': 0, 'observation': tensor([ 0.1301,  0.0152, -0.2024, -0.5679])}
{'action': tensor(1)}
{'time': tensor(21), 'reward': tensor(1.), 'done': tensor(1), 'episo

{'action': tensor(0)}
{'time': tensor(15), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 7, 'task_index': 0, 'observation': tensor([ 0.0455,  0.2506, -0.1268, -0.5969])}
{'action': tensor(1)}
{'time': tensor(16), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 7, 'task_index': 0, 'observation': tensor([ 0.0505,  0.4473, -0.1387, -0.9267])}
{'action': tensor(1)}
{'time': tensor(17), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 7, 'task_index': 0, 'observation': tensor([ 0.0594,  0.6440, -0.1573, -1.2595])}
{'action': tensor(1)}
{'time': tensor(18), 'reward': tensor(1.), 'done': tensor(0), 'episode_index': 7, 'task_index': 0, 'observation': tensor([ 0.0723,  0.8407, -0.1825, -1.5970])}
{'action': tensor(0)}
{'time': tensor(19), 'reward': tensor(1.), 'done': tensor(1), 'episode_index': 7, 'task_index': 0, 'observation': tensor([ 0.0891,  0.6482, -0.2144, -1.3664])}
{'action': tensor(1)}
{'time': tensor(0), 'reward': tensor(0.), 'done': tensor(0), 'episod

## Inspect specs

`input_spec` and `output_spec` describe the full contract before any steps. Use them to allocate buffers, build network heads, or validate shapes.

In [4]:
ispec = env.input_spec
ospec = env.output_spec

print("Input spec:")
print(f"  action  dtype={ispec.action.dtype}  shape={ispec.action.shape}")
print()
print("Output spec:")
print(f"  observation  dtype={ospec.observation.dtype}  shape={ospec.observation.shape}")
print(f"  reward       dtype={ospec.reward.dtype}        shape={ospec.reward.shape}")
print(f"  done         dtype={ospec.done.dtype}          shape={ospec.done.shape}")
print(f"  time         dtype={ospec.time.dtype}          shape={ospec.time.shape}")

Input spec:
  action  dtype=torch.int64  shape=()

Output spec:
  observation  dtype=torch.float32  shape=(4,)
  reward       dtype=torch.float32        shape=()
  done         dtype=torch.int64          shape=()
  time         dtype=torch.int64          shape=()


## Cleanup

In [5]:
env.close()